# CADI Deep Learning - Semana 12
# # Semana 12:  Implementación del Mecanismo de Atención para Series de Tiempo y un Transformer Básico en Google Colab 
# **Estudiante:** Jenifer Cárdenas Aguilera

## 1. Selección y Preparación de Datos
Utilizamos una serie de tiempo sintética (onda sinoidal con ruido y tendencia) para simular datos secuenciales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models

# Generación de datos secuenciales
time = np.arange(0, 400, 0.5)
data = 0.2 * time + 10 * np.sin(time) + np.random.randn(len(time))

# Normalización
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data.reshape(-1, 1))

# Preparación en ventanas temporales (Look-back de 10 pasos)
def crear_ventanas(dataset, ventana=10):
    X, y = [], []
    for i in range(len(dataset) - ventana):
        X.append(dataset[i:(i + ventana), 0])
        y.append(dataset[i + ventana, 0])
    return np.array(X), np.array(y)

ventana_tiempo = 10
X, y = crear_ventanas(data_scaled, ventana_tiempo)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# División en entrenamiento y prueba
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

plt.figure(figsize=(10, 4))
plt.plot(data[:100], label='Muestra de Serie Original')
plt.title('Selección de Datos: Serie de Tiempo con Estacionalidad')
plt.legend()
plt.show()

## 2. Implementación del Mecanismo de Atención y Modelo Transformer
Construimos una arquitectura tipo Transformer simplificada utilizando la capa de `MultiHeadAttention` para ponderar la importancia de los elementos de la secuencia.

In [ ]:
def transformer_model():
    inputs = layers.Input(shape=(ventana_tiempo, 1))
    
    # 1. Mecanismo de Atención (ponderación de elementos)
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=2)(inputs, inputs)
    attention = layers.Dropout(0.1)(attention)
    
    # 2. Conexión residual y normalización de capa
    x = layers.Add()([inputs, attention])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # 3. Capas densas de clasificación/regresión
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1)(x)
    
    return models.Model(inputs, outputs)

model = transformer_model()
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("Entrenamiento del modelo basado en atención...")
history = model.fit(X_train, y_train, epochs=25, batch_size=32, validation_data=(X_test, y_test), verbose=1)

## 3. Evaluación y Visualización de Resultados
Evaluamos el desempeño del modelo mediante métricas de error (MSE, MAE) y visualizamos las predicciones frente a los valores reales.

In [ ]:
pred = model.predict(X_test)
y_real_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_inv = scaler.inverse_transform(pred)

plt.figure(figsize=(10, 5))
plt.plot(y_real_inv, label="Valores Reales", color='darkblue')
plt.plot(y_pred_inv, label="Predicción Transformer", color='orange', linestyle='--')
plt.title("Evaluación: Comparación Valores Reales vs Predicciones")
plt.legend()
plt.show()

# Visualización de la pérdida
plt.plot(history.history['loss'], label='Loss Entrenamiento')
plt.plot(history.history['val_loss'], label='Loss Validación')
plt.title('Curva de Aprendizaje (MSE)')
plt.legend()
plt.show()

## 4. Análisis Comparativo y Conclusiones

### Comparación Conceptual con RNN
A diferencia del modelo RNN desarrollado anteriormente, que procesa la información de forma secuencial y puede 'olvidar' datos lejanos debido al desvanecimiento del gradiente, este modelo basado en **atención** permite ponderar cada paso de la ventana de forma independiente. Esto facilita la captura de dependencias tanto de corto como de largo plazo de manera simultánea.

### Análisis de Comportamiento
- **Dependencias de Corto Plazo**: El modelo identifica rápidamente cambios inmediatos en la tendencia.
- **Dependencias de Largo Plazo**: La atención permite que el modelo 'recuerde' patrones estacionales pasados sin degradación de la información.

### Conclusiones Técnicas
1. **Ventajas**: Los mecanismos de atención superan el cuello de botella secuencial, permitiendo un procesamiento paralelo y una mejor ponderación de la relevancia de los datos pasados.
2. **Limitaciones**: Requieren más parámetros y datos para converger adecuadamente en comparación con arquitecturas recurrentes más simples.
3. **Casos de Uso**: Son ideales para series de tiempo complejas como predicción financiera o de demanda, donde el contexto histórico es vital para el pronóstico futuro.